Setup:

In [1]:
import gc
import torch

torch.cuda.empty_cache()
gc.collect()

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B', dtype=torch.bfloat16, device_map="auto")
model.seqlen = model.config.max_position_embeddings

/home/mrajnoha/double-block-sparse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:11<00:00, 25.50it/s] 


In [3]:
from llm import find_layers

find_layers(model)

# length = 0
# for name, param in model.named_parameters():
#     if 'proj' in name:
#         print(name, param.shape)
#         length += 1
# print(f'Matrix count: {length}')

{'model.layers.0.self_attn.q_proj': Linear(in_features=4096, out_features=4096, bias=False),
 'model.layers.0.self_attn.k_proj': Linear(in_features=4096, out_features=1024, bias=False),
 'model.layers.0.self_attn.v_proj': Linear(in_features=4096, out_features=1024, bias=False),
 'model.layers.0.self_attn.o_proj': Linear(in_features=4096, out_features=4096, bias=False),
 'model.layers.0.mlp.gate_proj': Linear(in_features=4096, out_features=14336, bias=False),
 'model.layers.0.mlp.up_proj': Linear(in_features=4096, out_features=14336, bias=False),
 'model.layers.0.mlp.down_proj': Linear(in_features=14336, out_features=4096, bias=False),
 'model.layers.1.self_attn.q_proj': Linear(in_features=4096, out_features=4096, bias=False),
 'model.layers.1.self_attn.k_proj': Linear(in_features=4096, out_features=1024, bias=False),
 'model.layers.1.self_attn.v_proj': Linear(in_features=4096, out_features=1024, bias=False),
 'model.layers.1.self_attn.o_proj': Linear(in_features=4096, out_features=4096

In [4]:
model.hf_device_map

{'model.embed_tokens': 0,
 'model.layers.0': 0,
 'model.layers.1': 0,
 'model.layers.2': 0,
 'model.layers.3': 0,
 'model.layers.4': 0,
 'model.layers.5': 0,
 'model.layers.6': 0,
 'model.layers.7': 0,
 'model.layers.8': 0,
 'model.layers.9': 0,
 'model.layers.10': 0,
 'model.layers.11': 0,
 'model.layers.12': 0,
 'model.layers.13': 0,
 'model.layers.14': 1,
 'model.layers.15': 1,
 'model.layers.16': 1,
 'model.layers.17': 1,
 'model.layers.18': 1,
 'model.layers.19': 1,
 'model.layers.20': 1,
 'model.layers.21': 1,
 'model.layers.22': 1,
 'model.layers.23': 1,
 'model.layers.24': 1,
 'model.layers.25': 1,
 'model.layers.26': 1,
 'model.layers.27': 1,
 'model.layers.28': 1,
 'model.layers.29': 1,
 'model.layers.30': 1,
 'model.layers.31': 1,
 'model.norm': 1,
 'model.rotary_emb': 1,
 'lm_head': 1}

In [5]:
q_proj_weight = model.model.layers[3].self_attn.q_proj.weight
print(q_proj_weight.shape)   # torch.Size([4096, 4096])
print(q_proj_weight.device)
print(q_proj_weight.dtype)
print(q_proj_weight)

k_proj_weight = model.model.layers[5].self_attn.k_proj.weight
print(k_proj_weight.shape)   # torch.Size([1024, 4096])
print(k_proj_weight.device)
print(k_proj_weight.dtype)
print(k_proj_weight)

mlp_up_weight = model.model.layers[8].mlp.up_proj.weight
print(mlp_up_weight.shape)   # torch.Size([14336, 4096])
print(mlp_up_weight.device)
print(mlp_up_weight.dtype)
print(mlp_up_weight)


torch.Size([4096, 4096])
cuda:0
torch.bfloat16
Parameter containing:
tensor([[ 0.0092,  0.0076,  0.0015,  ...,  0.0166,  0.0034,  0.0076],
        [ 0.0125,  0.0153,  0.0129,  ..., -0.0004,  0.0128,  0.0056],
        [ 0.0192,  0.0037, -0.0167,  ...,  0.0115,  0.0078, -0.0008],
        ...,
        [-0.0164,  0.0254,  0.0026,  ...,  0.0089, -0.0062,  0.0029],
        [-0.0457,  0.0171, -0.0064,  ..., -0.0184, -0.0020, -0.0018],
        [-0.0135, -0.0003, -0.0234,  ..., -0.0082,  0.0088,  0.0267]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)
torch.Size([1024, 4096])
cuda:0
torch.bfloat16
Parameter containing:
tensor([[-0.0021,  0.0483, -0.0437,  ...,  0.0026, -0.0344,  0.0060],
        [ 0.0170,  0.0552,  0.0059,  ...,  0.0112,  0.0259,  0.0011],
        [-0.0035,  0.0186, -0.0403,  ...,  0.0056, -0.0273,  0.0025],
        ...,
        [ 0.0164, -0.0613,  0.0312,  ...,  0.0625,  0.0381, -0.0449],
        [ 0.0226, -0.0013,  0.0204,  ..., -0.0031, -0.0077,  0.0295],

Tests:

In [6]:
from test_dbsf import *

In [7]:
# a = test_double_sparse(matrix=q_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=1)

In [8]:
# b = test_hybrid(matrix=q_proj_weight, asp=0.25, bsp=0.25)

In [9]:
# c = test_2to4_A_B(matrix=q_proj_weight, sp=0.25, mid_dim_scale=1)

In [10]:
# d = test_1x2_2x1(matrix=q_proj_weight, sp=0.25, mid_dim_scale=1)

In [11]:
# e = test_mag_prune(matrix=q_proj_weight, sp=0.5)

In [12]:
# f = test_double_block_sparse(matrix=q_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=2)

In [ ]:
g = test_double_block_sparse(matrix=k_proj_weight, total_sp=0.5, b_bias=0.5, mid_dim_scale=1)

Error pre-finalization:  763.4482421875
Error post-finalization:  761.0419921875
Sparsity check:  0.4999617508479527
Frobenius norm: 27.586992263793945
A: tensor([[-0.0000e+00, -0.0000e+00, -9.0163e-05, -2.4558e-02],
        [ 0.0000e+00,  0.0000e+00,  2.1354e-02,  2.4633e-02],
        [ 9.3672e-03, -2.8629e-03, -0.0000e+00,  0.0000e+00],
        [ 9.2617e-03,  2.5844e-02, -0.0000e+00,  0.0000e+00]])
B: tensor([[ 0.8025, -0.0036,  0.0000,  0.0000],
        [ 0.0129,  0.7606, -0.0000,  0.0000],
        [ 0.0217, -0.0202,  0.7837,  0.0096],
        [-0.0207,  0.0281,  0.0183,  0.7781]])
A.size() = torch.Size([14336, 4096])
B.size() = torch.Size([4096, 4096])
AB: tensor([[-0.0139, -0.0058,  0.0026,  ...,  0.0122,  0.0149, -0.0051],
        [-0.0002,  0.0033,  0.0154,  ..., -0.0114,  0.0025,  0.0155],
        [ 0.0178, -0.0012,  0.0022,  ..., -0.0022,  0.0019, -0.0070],
        ...,
        [-0.0056,  0.0098,  0.0136,  ..., -0.0026, -0.0155, -0.0058],
        [ 0.0018, -0.0019, -0.0021,  .

In [14]:
# h = test_2to4_A_B(matrix=q_proj_weight, sp=0.5, mid_dim_scale=0.5)

Output:

In [15]:
import seaborn as sns
sns.reset_orig()

from matplotlib import rcParams
rcParams['figure.figsize'] = 12.5, 6

outputs = {'dsf': a,
           'hybrid': b, 
           '2:4': c,
           '1x2': d,
           'mag_prune': e,
           'dbsf_2x_mid': f,
           'dbsf': g,
           '2:4_compr': h,
           }
alpha = [1, 0.5, 0.5, 1, 1, 1, 1, 1]

bars = sns.barplot(x = outputs.keys(), y = outputs.values())
bars.tick_params(axis='x', labelsize=12)
for bar, a in zip(bars.containers[0], alpha):
    bar.set_alpha(a)

NameError: name 'a' is not defined

Cleanup:

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()